In [1]:
import sumolib

ruta_net = "../data/processed/usaquen.net.xml"

net = sumolib.net.readNet(ruta_net)
nodos = []
for node in net.getNodes():
    x, y = node.getCoord()
    lon, lat = net.convertXY2LonLat(x, y)
    nodos.append({"id": node.getID(), "lon": lon, "lat": lat})

print(len(nodos), "nodos cargados")

12989 nodos cargados


In [12]:
import requests
import time

URL = "https://serviciosgis.catastrobogota.gov.co/arcgis/rest/services/topografia/modelodigitalterrenobogotaurbano/MapServer/identify"

def consultar_elevacion(lon, lat, reintentos=3):
    params = {
        "geometry": f"{lon},{lat}",
        "geometryType": "esriGeometryPoint",
        "sr": 4326,
        "layers": "all",
        "tolerance": 1,
        "mapExtent": f"{lon-0.001},{lat-0.001},{lon+0.001},{lat+0.001}",
        "imageDisplay": "400,400,96",
        "returnGeometry": "false",
        "f": "json"
    }
    for intento in range(reintentos):
        try:
            r = requests.get(URL, params=params, timeout=20)
            data = r.json()
            if data.get("results"):
                val = data["results"][0]["attributes"].get("Stretch.Pixel Value")
                if val is None or val == "NoData":
                    return None
                try:
                    return float(val)
                except ValueError:
                    return None
            return None
        except requests.exceptions.RequestException:
            time.sleep(2)  # espera un poco antes de reintentar
    return None  # si fallan todos los reintentos, se rinde y devuelve None

In [3]:
import time
t0 = time.time()
z = consultar_elevacion(nodos[0]["lon"], nodos[0]["lat"])
print(z, time.time() - t0, "segundos")

2549.013916 0.5398752689361572 segundos


In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import json
import os

RUTA_CHECKPOINT = "../data/processed/elevaciones_nodos.json"

# Si ya existe un checkpoint de un intento anterior, cárgalo
if os.path.exists(RUTA_CHECKPOINT):
    with open(RUTA_CHECKPOINT, "r") as f:
        resultados = json.load(f)
    print("Checkpoint cargado:", len(resultados), "resultados previos")
else:
    resultados = {}

cache = {}

def clave(lon, lat):
    return (round(lon, 5), round(lat, 5))

def consultar_con_cache(n):
    k = clave(n["lon"], n["lat"])
    if k not in cache:
        try:
            cache[k] = consultar_elevacion(n["lon"], n["lat"])
        except Exception:
            cache[k] = None
    return n["id"], cache[k]

# Solo consulta los nodos que NO están ya en el checkpoint
pendientes = [n for n in nodos if n["id"] not in resultados]
print("Pendientes por consultar:", len(pendientes))

contador = 0
with ThreadPoolExecutor(max_workers=10) as executor:
    futuros = {executor.submit(consultar_con_cache, n): n for n in pendientes}
    for f in tqdm(as_completed(futuros), total=len(pendientes)):
        nid, z = f.result()
        resultados[nid] = z
        contador += 1
        if contador % 500 == 0:
            with open(RUTA_CHECKPOINT, "w") as fp:
                json.dump(resultados, fp)

# Guardado final
with open(RUTA_CHECKPOINT, "w") as fp:
    json.dump(resultados, fp)

for n in nodos:
    n["z"] = resultados.get(n["id"])

con_z = [n for n in nodos if n.get("z") is not None]
print("Nodos con z:", len(con_z), "de", len(nodos))

Pendientes por consultar: 12989


100%|██████████| 12989/12989 [34:53<00:00,  6.20it/s] 


Nodos con z: 12907 de 12989


In [6]:
import xml.etree.ElementTree as ET

tree = ET.parse("../data/processed/usaquen_plain.nod.xml")
root = tree.getroot()
z_por_id = {n["id"]: n["z"] for n in nodos if n.get("z") is not None}

encontrados = 0
for node in root.findall("node"):
    nid = node.get("id")
    if nid in z_por_id:
        node.set("z", str(z_por_id[nid]))
        encontrados += 1

print("Coincidencias encontradas:", encontrados, "de", len(z_por_id))
tree.write("../data/processed/usaquen_plain_z.nod.xml")

Coincidencias encontradas: 12907 de 12907


In [14]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import json, os

RUTA_CHECKPOINT_EDGES = "../data/processed/elevaciones_edges.json"

tree_edg = ET.parse("../data/processed/usaquen_plain.edg.xml")
root_edg = tree_edg.getroot()

# Recolectar TODOS los puntos únicos (lon, lat) de todos los shapes
puntos_unicos = {}  # clave: (lon5, lat5) -> None
edges_data = []

for edge in root_edg.findall("edge"):
    shape = edge.get("shape")
    if not shape:
        continue
    puntos = [tuple(map(float, p.split(","))) for p in shape.split(" ")]
    edges_data.append({"id": edge.get("id"), "puntos": puntos})
    for x, y in puntos:
        lon, lat = net.convertXY2LonLat(x, y)
        k = (round(lon, 5), round(lat, 5))
        puntos_unicos[k] = (lon, lat)

print("Edges con shape:", len(edges_data))
print("Puntos únicos a consultar:", len(puntos_unicos))

# Cargar checkpoint si existe
if os.path.exists(RUTA_CHECKPOINT_EDGES):
    with open(RUTA_CHECKPOINT_EDGES, "r") as f:
        cache_edges = json.load(f)
    cache_edges = {eval(k): v for k, v in cache_edges.items()}  # claves guardadas como string

    print("Checkpoint cargado:", len(cache_edges), "puntos ya resueltos")
else:
    cache_edges = {}

# Reusar lo que ya está en el cache de nodos (muchos coinciden)
for k in list(puntos_unicos.keys()):
    if k in cache and k not in cache_edges:
        cache_edges[k] = cache[k]

pendientes = [k for k in puntos_unicos if k not in cache_edges]
print("Pendientes por consultar:", len(pendientes))

def consultar_punto(k):
    lon, lat = puntos_unicos[k]
    try:
        return k, consultar_elevacion(lon, lat)
    except Exception:
        return k, None

contador = 0
with ThreadPoolExecutor(max_workers=10) as executor:
    futuros = {executor.submit(consultar_punto, k): k for k in pendientes}
    for f in tqdm(as_completed(futuros), total=len(pendientes)):
        k, z = f.result()
        cache_edges[k] = z
        contador += 1
        if contador % 1000 == 0:
            with open(RUTA_CHECKPOINT_EDGES, "w") as fp:
                json.dump({str(k): v for k, v in cache_edges.items()}, fp)

with open(RUTA_CHECKPOINT_EDGES, "w") as fp:
    json.dump({str(k): v for k, v in cache_edges.items()}, fp)


print("Total resuelto:", len(cache_edges))

Edges con shape: 18683
Puntos únicos a consultar: 44268
Pendientes por consultar: 21330


100%|██████████| 21330/21330 [53:35<00:00,  6.63it/s]   


Total resuelto: 44268


In [15]:
for edge in root_edg.findall("edge"):
    ed = next((e for e in edges_data if e["id"] == edge.get("id")), None)
    if ed is None:
        continue
    nuevos_puntos = []
    for x, y in ed["puntos"]:
        lon, lat = net.convertXY2LonLat(x, y)
        k = (round(lon, 5), round(lat, 5))
        z = cache_edges.get(k)
        if z is None:
            z = 0
        nuevos_puntos.append(f"{x},{y},{z}")
    edge.set("shape", " ".join(nuevos_puntos))

tree_edg.write("../data/processed/usaquen_plain_z.edg.xml")
print("Listo")

Listo
